# Model inference with Keras

Loads the `.keras` model and **StandardScaler** from `02_training_keras`, and the **LabelEncoder** from data preparation (`label_encoding.pkl`). Runs on `data/inference/input/input.csv`, checks predictions against `data/inference/expected/expected.csv`, and writes `data/inference/keras/output_keras.csv`.


## Import libraries


In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf


## Config


In [ ]:
from config import (
    FEATURE_COLS,
    KERAS_INFERENCE_DIR,
    KERAS_INFERENCE_EXPECTED_PATH,
    KERAS_INFERENCE_INPUT_PATH,
    KERAS_INFERENCE_LABEL_ENCODING_PATH,
    KERAS_INFERENCE_NATIVE_KERAS_FILE_PATH,
    KERAS_INFERENCE_OUTPUT_PATH,
    KERAS_INFERENCE_SCALER_JOBLIB_PATH,
)


## Load Keras model


In [ ]:
model_path=KERAS_INFERENCE_NATIVE_KERAS_FILE_PATH

model = tf.keras.models.load_model(model_path)
print(f"Model loaded from: {model_path}")
model.summary()


## Load scaler and label encoding


In [ ]:
scaler_path=KERAS_INFERENCE_SCALER_JOBLIB_PATH
label_encoding_path=KERAS_INFERENCE_LABEL_ENCODING_PATH

scaler = joblib.load(scaler_path)
label_encoder = joblib.load(label_encoding_path)
print(f"Scaler: {scaler_path}")
print(f"Label encoding: {label_encoding_path}")
print("Classes:", label_encoder.classes_)


## Load inference input


In [ ]:
input_path=KERAS_INFERENCE_INPUT_PATH

df_input = pd.read_csv(input_path)
df_input[FEATURE_COLS] = df_input[FEATURE_COLS].astype(np.float32)
print(f"Input rows: {len(df_input)}")
df_input.head()


# Transform Input

In [ ]:
X = df_input[FEATURE_COLS].values
X_scaled = scaler.transform(X)


## Run inference


In [ ]:
proba = model.predict(X_scaled, verbose=0)

y_pred = np.argmax(proba, axis=1)
species_pred = label_encoder.inverse_transform(y_pred)

df_out = df_input.copy()
df_out["species"] = np.asarray(species_pred, dtype=str)
df_out.head()


## Verify against expected.csv


In [ ]:
expected_path=KERAS_INFERENCE_EXPECTED_PATH

df_expected = pd.read_csv(expected_path)
expected = df_expected["species"].astype(str).to_numpy()
predicted = df_out["species"].astype(str).to_numpy()

if np.array_equal(predicted, expected):
    print("Verification: predictions match expected.csv")
else:
    print("Verification: predictions do not match expected.csv")
    print("Expected:", expected)
    print("Got:     ", predicted)


## Save predictions


In [ ]:
output_dir=KERAS_INFERENCE_DIR
os.makedirs(output_dir, exist_ok=True)

df_out.to_csv(KERAS_INFERENCE_OUTPUT_PATH, index=False)
print(f"Saved: {KERAS_INFERENCE_OUTPUT_PATH}")
